In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import glob
from astropy.io import fits
from matplotlib.widgets import Slider
from astropy.table import Table
import astropy.units as u
from astropy import constants as const

from reproject import reproject_interp
from scipy.ndimage import fourier_shift
from skimage.registration import phase_cross_correlation
from Functions import *
from astropy.io import fits
from astropy.wcs import WCS
from reproject import reproject_interp as rpj
from astropy.convolution import convolve, convolve_fft
from scipy.ndimage import zoom, shift as ndi_shift
from photutils.centroids import centroid_quadratic
import time
from photutils.aperture import CircularAperture, CircularAnnulus, aperture_photometry
from astropy.wcs.utils import proj_plane_pixel_area

%matplotlib widget

from ImageScience import ImageScience
ngc1672 = ImageScience()

files = glob.glob('/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1672/*log.fits')
names = ['ha_cont_err', 'ha_cont', 'ha_contsub_err', 'ha_contsub', 'ha_nii_err', 'ha_nii']
for i, file in enumerate(files):
    ngc1672.load_image(names[i], file)
ngc1672.load_image('pa_contsub', '/project/galaxies/tjuchau/data_files/JWST/PHANGS_products/hlsp_4793_jwst_nircam_ngc1672_f187n_v4p1_line-atf150wxf187nxf200w.fits', hdu="CONTSUB")
ngc1672.load_image('pa_cont', '/project/galaxies/tjuchau/data_files/JWST/PHANGS_products/hlsp_4793_jwst_nircam_ngc1672_f187n_v4p1_line-atf150wxf187nxf200w.fits', hdu="CONTINUUM")
ngc1672.sum_images('pa_cont', 'pa_contsub', out_name='pa_full')
ngc1672.sum_images('ha_cont', 'ha_contsub', out_name='ha_full')
print(f'Image object created with keys for {ngc1672.images.keys()}')

table = Table.read('/project/galaxies/tjuchau/data_files/Kiana_Cluster_Files/full_table_ryan.csv')
table.add_column([0]*len(table), name = 'pa_EW')
table.add_column([0]*len(table), name = 'test1')
table.add_column([0]*len(table), name = 'test2')
table.add_column([0]*len(table), name = 'test3')
table.add_column([0]*len(table), name = 'ha_EW')
table = table[table['galaxy']=="ngc1672"]


In [ ]:
ngc1672.display(['ha_cont', 'ha_contsub', 'pa_cont', 'pa_contsub'], [table[0]['ra'], table[0]['dec']], 0.2*u.arcsec, background_annulus_thickness=0.1*u.arcsec, buffer=0.1*u.arcsec, ncols=2)

In [ ]:
for row in table:
    ha_ew = ngc1672.get_equivalent_width('ha_full', 'ha_cont', [row['ra'], row['dec']], 0.2*u.arcsec, 0.1*u.arcsec, buffer=0.3*u.arcsec)
    ha_ew_nobg = ngc1672.get_equivalent_width('ha_full', 'ha_cont', [row['ra'], row['dec']], 0.2*u.arcsec, 0*u.arcsec, buffer=0*u.arcsec)
    pa_ew = ngc1672.get_equivalent_width('pa_full', 'pa_cont', [row['ra'], row['dec']], 0.2*u.arcsec, 0.1*u.arcsec, buffer=0.3*u.arcsec)
    pa_ew_nobg = ngc1672.get_equivalent_width('pa_full', 'pa_cont', [row['ra'], row['dec']], 0.2*u.arcsec, 0*u.arcsec, buffer=0*u.arcsec)
    row['ha_EW'] = ha_ew[0].value
    row['pa_EW'] = pa_ew[0].value
    row['test1'] = pa_ew_nobg[0].value
    row['test2'] = ha_ew_nobg[0].value


In [ ]:
plt.clf()
plt.scatter(table['test2'], table['test1'])
plt.xlabel('H-alpha')
plt.ylabel('Pa-alpha')
plt.ylim([np.min(table['pa_EW']), 6000])

plt.show()

In [ ]:
plt.clf()
plt.scatter(table['ha_EW'], table['pa_EW'])
plt.xlabel('H-alpha')
plt.ylabel('Pa-alpha')
plt.ylim([np.min(table['pa_EW']), 6000])

plt.show()

In [ ]:
pa_ew = ngc1672.get_equivalent_width('pa_full', 'pa_cont', [table[table['pa_EW']>14000]['ra'][0], table[table['pa_EW']>14000]['dec'][0]], 0.2*u.arcsec, 0.1*u.arcsec, buffer=0.3*u.arcsec)
cont_est = ngc1672.get_background_subtracted_flux('pa_cont', [table[table['pa_EW']>14000]['ra'][0], table[table['pa_EW']>14000]['dec'][0]], 0.2*u.arcsec, 0.1*u.arcsec, buffer=0.3*u.arcsec)
full_flux = ngc1672.get_background_subtracted_flux('pa_full', [table[table['pa_EW']>14000]['ra'][0], table[table['pa_EW']>14000]['dec'][0]], 0.2*u.arcsec, 0.1*u.arcsec, buffer=0.3*u.arcsec)


In [ ]:
ngc1672.display(['pa_full', 'pa_cont', 'pa_contsub'], 
[table[table['pa_EW']>14000]['ra'][0], table[table['pa_EW']>14000]['dec'][0]], 
0.2*u.arcsec, background_annulus_thickness=0.1*u.arcsec, buffer = 0.3*u.arcsec,ncols=2)

In [ ]:
ngc1672.get_background_subtracted_flux('pa_full', 
[table[table['pa_EW']>14000]['ra'][0], table[table['pa_EW']>14000]['dec'][0]], 
0.2*u.arcsec, background_annulus_thickness=0.1*u.arcsec, buffer = 0.3*u.arcsec)

In [ ]:
ngc1672.get_background_subtracted_flux('pa_full', 
[table[table['pa_EW']>14000]['ra'][0], table[table['pa_EW']>14000]['dec'][0]], 
0.2*u.arcsec, background_annulus_thickness=0.1*u.arcsec, buffer = 0.1*u.arcsec)

In [ ]:
ngc1672.align_images('pa_cont', 'ha_cont')
ngc1672.align_images('pa_contsub', 'ha_contsub')
ngc1672.make_ew_ratio_image('ha_cont_aligned', 'ha_contsub_aligned', 'pa_cont', 'pa_contsub', 
output_name='EW_Ha_over_PaA', min_continuum=0, min_line=-np.inf)
ngc1672.save_fits('EW_Ha_over_PaA', '/project/galaxies/tjuchau/data_files/HST/HST_reduced_images/ngc1672/Ha_over_Pa_EW_ratio.fits')